# Government QA Data Processing

Public portfolio edition prepared for GitHub and Databricks. Credentials are read from environment variables; research data and generated artifacts are not committed to Git.


In [ ]:
# Databricks uses Unity Catalog Volumes; no Google Drive mount is required.

In [ ]:
from pathlib import Path
QS_DIR = Path("/Volumes/main/default/thesis_project/data/government_qa/gov-report-qs")

In [ ]:
#1. Configure path: your GovReport-QS directory (containing *.jsonl files)
jsonl_files = sorted(QS_DIR.glob("*.jsonl"))
assert jsonl_files, f"No *.jsonl files found under {QS_DIR}"
print("JSONL files found:", len(jsonl_files))
for p in jsonl_files[:10]:
    print(" -", p.name)

In [ ]:
# Identify split from filename and keep (split, path) pairs
split_files = []
for p in jsonl_files:
    name = p.stem.lower()
    if name.startswith("train"):
        sp = "train"
    elif name.startswith("test"):
        sp = "test"
    elif name.startswith("valid") or name.startswith("val"):
        sp = "valid"
    else:
        sp = "unknown"
    split_files.append((sp, p))


In [ ]:
print("\nSplits detected:")
for sp, p in split_files:
    print(f" - {sp}: {p.name}")

In [ ]:
#2. Utilities: read JSONL, split sample_id, flatten questions, collect evidence
import json, pandas as pd
from typing import List, Dict, Any, Optional

In [ ]:
def load_jsonl(path: Path):
    """
    Stream a JSONL file line by line and yield parsed JSON objects.
    Raises a helpful error when a specific line cannot be parsed.
    """
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                yield json.loads(line)
            except Exception as e:
                raise ValueError(f"{path.name} line {i} JSON parsing failed: {e}")

In [ ]:
def split_sample_id(sample_id: str):
    """
    Split sample_id formatted as {reportID}_{summaryParagraphID}.
    Use the LAST underscore to avoid breaking report IDs that also contain underscores.
    Returns (report_id, summary_para_id).
    """
    if "_" not in sample_id:
        return sample_id, None
    i = sample_id.rfind("_")
    return sample_id[:i], sample_id[i+1:]


In [ ]:
def flatten_questions(nodes: List[Dict[str, Any]], sample_id: str):
    """
    Flatten the hierarchical `questions` into multiple rows.
    Each node (including child questions) becomes one record.
    Fields: qid, parent_qid, depth, path, question, gold_answer.
    If `answer` is a list, join elements with spaces to form a string.
    """
    rows = []
    counter = 0
    def rec(lst, parent_qid: Optional[str], depth: int, path: List[int]):
        nonlocal counter
        if not isinstance(lst, list):
            return
        for idx, q in enumerate(lst):
            counter += 1
            qid = f"{sample_id}#q{counter}"
            question = (q.get("question") or "").strip()
            ans = q.get("answer")
            # Normalize answer to string
            if isinstance(ans, list):
                ans = " ".join([str(a).strip() for a in ans if str(a).strip()])
            answer = (ans or "").strip()
            rows.append({
                "sample_id": sample_id,
                "qid": qid,
                "parent_qid": parent_qid,
                "depth": depth,
                "path": "/".join(map(str, path+[idx])),
                "question": question,
                "gold_answer": answer,
            })
            rec(q.get("child_questions") or [], qid, depth+1, path+[idx])
    rec(nodes or [], None, 0, [])
    return rows

In [ ]:
def collect_evidence(section_obj: Dict[str, Any]):
    """
    Collect 'oracle evidence' from the aligned `section` tree.
    - Recursively gather all paragraphs (tolerates both 'paragraphs' and misspelled 'paragrpahs').
    - Return: concatenated evidence text, paragraph count, unique list of section titles.
    """
    paragraphs, titles = [], []
    def walk(node):
        if not isinstance(node, dict):
            return
        title = (node.get("section_title") or "").strip()
        if title:
            titles.append(title)
        paras = node.get("paragraphs")
        if paras is None:
            paras = node.get("paragrpahs")  # tolerate misspelling
        if isinstance(paras, list):
            for p in paras:
                ptxt = (p or "").strip()
                if ptxt:
                    paragraphs.append(ptxt)
        subs = node.get("subsections")
        if isinstance(subs, list):
            for sub in subs:
                walk(sub)
    if isinstance(section_obj, dict):
        walk(section_obj)
    evidence_text = "\n".join(paragraphs).strip()
    titles_uniq = list(dict.fromkeys([t for t in titles if t]))
    return evidence_text, len(paragraphs), titles_uniq

In [ ]:
# 3 Parse files/lines to build a raw detail table (one row per question)
all_rows = []
raw_issues = []

In [ ]:
for split_name, file in split_files:
    for obj in load_jsonl(file):
        # sample_id
        meta = obj.get("metadata") or {}
        sample_id = meta.get("sample_id") or obj.get("sample_id")
        if not sample_id:
            raw_issues.append((file.name, "missing sample_id"))
            continue
        report_id, summary_para_id = split_sample_id(sample_id)

        # expand questions/answers
        q_rows = flatten_questions(obj.get("questions"), sample_id)

        # extract evidence from aligned section
        evidence_text, para_cnt, titles = collect_evidence(obj.get("section"))

        # one row per question, attach split
        for r in q_rows:
            all_rows.append({
                "split": split_name,
                "source_file": file.name,
                "sample_id": sample_id,
                "report_id": report_id,
                "summary_para_id": summary_para_id,
                "qid": r["qid"],
                "depth": r["depth"],
                "path": r["path"],
                "question": r["question"],
                "gold_answer": r["gold_answer"],
                "oracle_evidence": evidence_text,
                "evidence_paragraph_count": para_cnt,
                "evidence_section_titles": titles,
            })

In [ ]:
raw_df = pd.DataFrame(all_rows)
print("\nRaw expanded row count (one row per question):", raw_df.shape)
print("Raw rows per split:\n", raw_df["split"].value_counts(dropna=False))

In [ ]:
# 4 Basic structure/integrity checks
def non_empty(s):
    return isinstance(s, str) and s.strip() != ""

In [ ]:
stats = {
    "total_question_rows": int(len(raw_df)),
    "empty_question_rows": int((~raw_df["question"].apply(non_empty)).sum()),
    "empty_gold_answer_rows": int((~raw_df["gold_answer"].apply(non_empty)).sum()),
    "empty_oracle_evidence_rows": int((~raw_df["oracle_evidence"].apply(non_empty)).sum()),
    "zero_paragraph_evidence_rows": int((raw_df["evidence_paragraph_count"] <= 0).sum()),
}
print("\nIntegrity stats:", stats)


In [ ]:
# 5 Cleaning & unification: question / gold_answer / oracle_evidence
clean_df = raw_df.copy()

In [ ]:
# trim whitespace
for col in ["question", "gold_answer", "oracle_evidence"]:
    clean_df[col] = clean_df[col].fillna("").apply(lambda s: s.strip())


In [ ]:
# keep rows that have BOTH a gold answer AND non-empty evidence
mask = clean_df["gold_answer"].apply(non_empty) & clean_df["oracle_evidence"].apply(non_empty)
clean_df = clean_df[mask].reset_index(drop=True)

In [ ]:
# optional: drop duplicates (same report_id + question + answer + evidence)
clean_df = clean_df.drop_duplicates(
    subset=["report_id", "question", "gold_answer", "oracle_evidence"]
).reset_index(drop=True)

In [ ]:
print("\nRows after cleaning (combined):", len(clean_df))
print("Rows after cleaning per split:\n", clean_df["split"].value_counts())

In [ ]:
# 6 Compact previews to confirm final data shape
pd.set_option("display.max_colwidth", 200)

In [ ]:
preview_cols = ["split", "sample_id", "report_id", "qid", "question", "gold_answer", "evidence_paragraph_count"]
for sp in ["train", "valid", "test", "unknown"]:
    df_sp = clean_df[clean_df["split"] == sp]
    if not df_sp.empty:
        print(f"\n=== Preview: {sp} (showing up to 5 rows) ===")
        display(df_sp[preview_cols].head(5))

In [ ]:
# Also show one full example (truncated) for quick sanity check
if not clean_df.empty:
    example = clean_df.sample(1, random_state=42)[
        ["split", "question", "gold_answer", "oracle_evidence", "evidence_section_titles"]
    ]
    print("\n=== One full example (truncated columns) ===")
    display(example)

In [ ]:
# 7 Export to timestamped folder under the specified Drive path
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo

In [ ]:
# Base folder where the new timestamped subfolder will be created
SAVE_BASE = Path("/Volumes/main/default/thesis_project/data/government_qa/gov-report-qs")


In [ ]:
# Use Europe/Copenhagen time for the timestamp (optional but explicit)
now_cph = datetime.now(ZoneInfo("Europe/Copenhagen"))
stamp = now_cph.strftime("%Y%m%d_%H%M%S")

In [ ]:
# Create a new timestamped output directory, e.g., processed_20250804_094215
OUT_DIR = SAVE_BASE / f"processed_{stamp}"
OUT_DIR.mkdir(exist_ok=True, parents=True)

In [ ]:
# Save combined cleaned data
clean_df.to_parquet(OUT_DIR / "qs_all_qa_evidence.parquet", index=False)
clean_df.to_csv(OUT_DIR / "qs_all_qa_evidence.csv", index=False)

In [ ]:
# Save per-split files
for sp in sorted(clean_df["split"].dropna().unique()):
    df_sp = clean_df[clean_df["split"] == sp]
    df_sp.to_parquet(OUT_DIR / f"qs_{sp}_qa_evidence.parquet", index=False)
    df_sp.to_csv(OUT_DIR / f"qs_{sp}_qa_evidence.csv", index=False)

In [ ]:
# List saved files for confirmation
print("\nSaved to:", OUT_DIR)
for p in sorted(OUT_DIR.glob("*")):
    print(" -", p.name)